In [9]:
import subprocess
import os

result = subprocess.run('bash -c "source /etc/network_turbo && env | grep proxy"', shell=True, capture_output=True, text=True)
output = result.stdout
for line in output.splitlines():
    if '=' in line:
        var, value = line.split('=', 1)
        os.environ[var] = value

In [2]:
!nvidia-smi

Wed Mar 26 23:26:29 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.67                 Driver Version: 550.67         CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L20                     On  |   00000000:6F:01.0 Off |                  Off |
| N/A   27C    P8             33W /  350W |       1MiB /  49140MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Reproduction of Deepseek Aha moment with CountDown Game using GRPO

## Ref
- [training/mini-deepseek-r1-aha-grpo.ipynb](https://github.com/philschmid/deep-learning-pytorch-huggingface/blob/main/training/mini-deepseek-r1-aha-grpo.ipynb)

## Aha Moment
Aha moment is introduced by Deepseek when they use *pure RL* to train the model. In the paper they described an "aha moment" when using pure RL to train the model. During this phase, DeepSeek-R1-Zero (the first test of DeepSeek-R1) learns to allocate more thinking time to a problem by reevaluating its initial approach without any human feedback or data describing how to do it. They describe this as an "aha moment" as:

> This behavior is not only a testament to the model’s growing reasoning abilities but also a captivating example of how reinforcement learning can lead to unexpected and sophisticated outcomes.

## Countdown Game

The Countdown game is a numbers puzzle where players use a set of randomly drawn numbers and basic arithmetic operations (+, -, ×, ÷) to reach or get as close as possible to a target number.

> Target Number: 952
> Available Numbers: 25, 50, 75, 100, 3, 6
>
> (100 × (3 × 3)) + (50 + 6 / 3) = 952

## GRPO

Compared to PPO, *GRPO eliminates the value function* and *estimates the advantage in a group-relative manner*. For a specific question-answer pair $(q,a)$, the behavior policy $\pi_{\theta_{old}}$ samples a group of $G$ individual responses: $\{o_{i}\}_{i=1}^G$. The advantage of the $i$-th response is calculated by normalizing the group-level rewards $\{R_{i}\}_{i=1}^G$:
$$
\hat{A}_{i}=\frac{r_{i}-\text{mean}(\{R_{i}\}_{i=1}^G)}{\text{std}(\{R_{i}\}_{i=1}^G)}
$$
Similar to PPO, GRPO adopts a clipped objective, together with a directly imposed KL penalty term:
$$
\begin{align}
\mathcal{J}_{\text{GRPO}}(\theta)&=\mathbb{E}_{(q,a)\sim \mathcal{D},\{o_{i}\}_{i=1}^G\sim \pi_{old}(\cdot|q)} \\
&\left[ \frac{1}{G}\sum_{i=1}^G \frac{1}{|o_{i}|}\sum_{t=1}^{|o_{i}|}\left( \min\left( r_{i,t}(\theta) \hat{A}_{i,t}, \text{clip} \left(r_{i,t}(\theta),1-\epsilon,1+\epsilon\right)\hat{A}_{i,t}\right)-\beta D_{\text{KL}}(\pi_{\theta}||\pi_{ref})\right)\right]
\end{align}
$$
where
$$
r_{i,t}(\theta)=\frac{\pi_{\theta}(o_{i,t}|q, o_{i<t})}{\pi_{\theta_{old}}(o_{i,t}|q, o_{i<t})}
$$
It is also worth noting that GRPO computes the objective at the **sample-level**. To be exact, GRPO *first calculates the mean loss within each generated sequence*, *before averaging the loss of different samples*.  

### GRPO Steps

- Sampling: Generate multiple outputs for each prompt using the current policy
- Reward Scoring: Each generation is scored using a reward function, could be (rule-based or outcome-based)
- Advantage Calculation: The average reward of the generated outputs is used as a baseline. The advantage of each solution within the group is then computed relative to this baseline. The reward is normalized within a group.
- Policy Optimization: The policy tries to maximize the GRPO objective, which includes the calculated advantages and a KL divergence term. This is different from how PPO implements the KL term within the reward.

## 1. Setup Enviroment

In [3]:
# # Install Pytorch & other libraries, make sure to match your GPU driver version
# %pip install "torch==2.5.1" tensorboard "setuptools<71.0.0"  --index-url https://download.pytorch.org/whl/cu121

# # Install flash-attn
# %pip install flash-attn 

# # Install Hugging Face libraries
# %pip install  --upgrade \
#   "transformers==4.48.1" \
#   "datasets==3.1.0" \
#   "accelerate==1.3.0" \
#   "hf-transfer==0.1.9" \
#   "deepspeed==0.15.4" \
#   "trl==0.14.0"

# # install vLLM 
# %pip install "vllm==0.7.0"

# ## IMPORTANT: If you want to run the notebook and the interactive cells you also need to install the following libraries:
# # But first read it the blog post and then decide as they might conflict with the libraries for distributed training. 
# # %pip install "peft==0.14.0" "bitsandbytes==0.45.0"

## 2. Generate training samples with reasoning prefix from the Countdown Game

- Dataset: [Jiayi-Pan/Countdown-Tasks-3to4](https://huggingface.co/datasets/Jiayi-Pan/Countdown-Tasks-3to4)
- Model: [Qwen2.5-3B-Instruct](https://huggingface.co/Qwen/Qwen2.5-3B-Instruct)

In [16]:
from transformers import AutoTokenizer
from datasets import load_dataset

" Load dataset " 
dataset_id = "Jiayi-Pan/Countdown-Tasks-3to4"
dataset = load_dataset(dataset_id, split='train')
dataset = dataset.shuffle(seed=5525).select(range(50000))

" Load tokenizer "
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")

" generate r1 prompt with a prefix for the model to already start with the thinking process "
def generate_r1_prompt(numbers, target):
    r1_prefix = [
        {
            "role": "system",
            "content": "You are a helpful assistant. You first thinks about the reasoning process in the mind and then provides the user with the answer."
        },
        {
            "role": "user",
            "content": f"Using the numbers {numbers}, create an equation that equals {target}. You can use basic arithmetic operations (+, -, *, /) and each number can only use once. Show your work in <think></think> tags. And return the final equation and answer in <answer></answer> tags, for example <answer> (1 + 2) / 3 = 1 </answer>." 
        },
        {
            "role": "assistant",
            "content": "Let me solve this step by step. \n<think>"
        },
    ]
    return {"prompt": tokenizer.apply_chat_template(r1_prefix, tokenize=False, continue_final_message=True), "target": target}

# convert our dataset to the r1 prompt
dataset = dataset.map(lambda x: generate_r1_prompt(x["nums"], x["target"]))

# split the dataset into train and test
train_test_split = dataset.train_test_split(test_size=0.1)

train_dataset = train_test_split["train"]
test_dataset = train_test_split["test"]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [17]:
train_dataset, test_dataset

(Dataset({
     features: ['target', 'nums', 'prompt'],
     num_rows: 45000
 }),
 Dataset({
     features: ['target', 'nums', 'prompt'],
     num_rows: 5000
 }))

In [15]:
# train_dataset[0]

## 3. Train the model using GRPO

TRL supports Group Relative Policy Optimization (GRPO) through a dedicated `GRPOTrainer` for aligning LLMs from preference data. The `GRPOTrainer` is a subclass of the `Trainer` from the `transformers` library and supports all the same features, including logging, checkpointing, distributed training, and parameter efficient fine-tuning (PEFT).

The `GRPOTrainer` supports *generic Outcome Reward Models (ORM)* and *custom reward functions*, that can be used to implement Rule-Based Reward Models. In the Deepseek R1 paper they implemented Rule-Based Reward Models to verify the correctness of the generated solutions. In our exmaple we are going to do a similar approach, where we will create 2 reward functions that:

- `Format Reward`: Checks if the generated format is correct `<think> [thinking] </think><answer> [answer] </answer>`
- `Accuracy Reward`: Extracts the equation from the `<answer>` tag and evaluates it against the target and if every number is used once.

Note: Correct `<answer>` in our example includes the equation, for example `<answer> 55 + 36 - 7 - 19 </answer>`.

In [18]:
import re

def format_reward_func(completions, target, **kwargs):
    """
    Format: <think>...</think><answer>...</answer>
    args:
        completion: generated results
        target: label
    return:
        score(float)
    """
    rewards = []

    for completion, gt in zip(completions, target):
        try:
            # add synthetic <think> as its already part of the prompt 生成的回答在实际对话中往往已经由 prompt 预先填充了 <think> 部分, and
            # prefilled for the assistant to more easily match the regex  添加 <think> 避免整个匹配失败
            completion = '<think>' + completion
            regex = r"^<think>([^<]*(?:<(?!/?think>)[^<]*)*)<\/think>\n<answer>([\s\S]*?)<\/answer>$"
            match = re.search(regex, completion, re.DOTALL)

            if match is None or len(match.groups()) != 2:  # re 会捕捉到 两个捕获组 <think> 和 <answer>
                rewards.append(0.0)
            else:
                rewards.append(1.0)
        except Exception:
            rewards.append(0.0)
    return rewards


def equation_reward_func(completions, target, nums, **kwargs):
    """
    Evaluates completions based on:
    1. each number is used once
    2. Mathematical correctness of the answer

    Args:
        completions (list[str]): Generated outputs
        target (list[str]): Expected answers
        nums (list[str]): Available numbers
    
    Returns:
        list[float]: Reward scores
    """
    rewards = []

    for completion, gt, num in zip(completions, target, nums):
        try:
            completion = '<think>' + completion
            # Check if the format is correct
            match = re.search(r"<answer>(.*?)<\/answer>", completion)
            if match is None:
                rewards.append(0.0)
                continue
                
            # extract equation
            equation = match.group(1).strip()
            # extract all numbers in equation
            used_nums = [int(n) for n in re.findall(r'\d+', equation)]

            # check if all numbers are only used once
            if sorted(used_nums) != sorted(num):
                rewards.append(0.0)
                continue

            # Define a regex pattern that only allows numbers, operators, parentheses, and whitespace
            allowed_pattern = r'^[\d+\-*/().\s]+$'
            if not re.match(allowed_pattern, equation):
               rewards.append(0.0)
               continue
                
            # Evaluate the equation with restricted globals and locals
            result = eval(equation, {"__builti'ns__": None}, {})
            # check if the equation is correct and matches the ground truth
            if abs(float(result) - float(gt)) < 1e-5:
                rewards.append(1.0)
            else:
                rewards.append(0.0)
        except Exception:
            rewards.append(0.0)
    return rewards

In [19]:
# Test Reward funcs
correct_sample_1 = """We need to find an equation using the numbers 19, 36, 55, and 7
exactly once, with basic arithmetic operations, that equals 65. One possible
combination is 55 + 36 - 19 + 7... </think>
<answer> 55 + 36 - 7 - 19 </answer>"""

correct_sample_2 = """ ... </think>
<answer> 55 + 36 - 7 - 19 </answer>"""

wrong_format = """User: Using the numbers [19, 36, 55, 7], create an equation that equals 65."""

wrong_format_2 = """To find the equation that equals 79 using the numbers 95, 78, 6, 88, I'll start by adding 88 and 95:                      
95 + 88 = 183                                                                                                              
Now, let's subtract 104 from 183 to get 79:
183 - 104 = 79
<think> 183 - 104 = 79 </think><think> 183 - 104 = 79 </think><answer> 183 - 104 = 79 </answer>"""

wrong_result = """ ... </think>
<answer> 55 + 36 - 7 - 18 </answer>"""


test_rewards = format_reward_func(completions=[correct_sample_1, correct_sample_2, wrong_format, wrong_format_2, wrong_result], target=["65", "65", "65", "65", "65"], nums=[[19, 36, 55, 7]] * 5)
assert test_rewards == [1.0, 1.0, 0.0, 0.0, 1.0], "Reward function is not working"
test_rewards = equation_reward_func(completions=[correct_sample_1, correct_sample_2, wrong_format, wrong_format_2, wrong_result], target=["65", "65", "65", "65", "65"], nums=[[19, 36, 55, 7]] * 5)
assert test_rewards == [1.0, 1.0, 0.0, 0.0, 0.0], "Reward function is not working"

In [20]:
from trl import GRPOConfig, GRPOTrainer, get_peft_config, ModelConfig

# our model we are going to use as policy 
model_config = ModelConfig(
    model_name_or_path="Qwen/Qwen2.5-3B-Instruct",
    torch_dtype='bfloat16',
    attn_implementation='flash_attention_2',
    use_peft=True,
    load_in_4bit=True,
)

training_args = GRPOConfig(
    output_dir='./qwen-r1-aha/',
    learning_rate=5e-7,
    lr_scheduler_type='cosine',
    logging_steps=10,
    max_steps=100,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    bf16=True,
    # GRPO configs
    max_prompt_length=256,
    max_completion_length=1024, # max length of the generated output for solution
    num_generations=2,
    beta=0.001,
    
)

trainer = GRPOTrainer(
    model=model_config.model_name_or_path,
    reward_funcs=[format_reward_func, equation_reward_func],
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    peft_config=get_peft_config(model_config),
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
# Train and push the model to the Hub
trainer.train()
# Save model
trainer.save_model(training_args.output_dir)

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
